In [28]:
# qcqp_gnn_notebook_final.py
# Notebook-compatible. Saves outputs like the reference classification pipeline:
#   gnn_outputs/<RUN_TAG>/
#       results_gnn_<RUN_TAG>.csv      <- metrics summary
#       preds_<RUN_TAG>.csv            <- per-instance predictions
#       cm_<RUN_TAG>.png               <- confusion matrix (direct LP/SDP decision)
#       epoch_log_<RUN_TAG>.csv        <- per-epoch train/val metrics
#       config_<RUN_TAG>.csv           <- hyperparameters
#
# ARCHITECTURE NOTE:
#   The model directly predicts LP(1) vs SDP(0) in a single head.
#   Unbounded SDP instances are labelled LP=1 (true label only; not predicted separately).
#   This matches the classification and regression pipelines exactly.
#
# Usage:
#   cfg = RunConfig(train_pkl="...", train_csv="...", ...)
#   model, metrics = main(cfg)

import os
import re
import pickle
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Subset

from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool
from torch_geometric.utils import scatter

from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    balanced_accuracy_score, f1_score,
)


# ============================================================
# 0) RunConfig
# ============================================================

@dataclass
class RunConfig:
    # Data
    train_pkl: str = "instances_5_10_10.pkl"
    train_csv: Optional[str] = "instancesobj_5_10_10.csv"
    test_pkl: Optional[str] = "instances_5_1_50.pkl"
    test_csv: Optional[str] = "instancesobj_5_1_50.csv"
    cache_dir: Optional[str] = None

    # Output
    output_root: str = "gnn_outputs"

    # Model
    epochs: int = 10
    hidden: int = 64
    layers: int = 3
    dropout: float = 0.1
    lr: float = 1e-3
    weight_decay: float = 1e-5
    lambda_delta: float = 0.25   # weight on delta MSE loss (comparable instances only)

    # Batching
    batch_train: int = 8
    batch_eval: int = 16

    # Graph construction
    # eps_prime: threshold for LP-vs-SDP label (same as TAU_BINARY in clf, TAU_SIGN in reg)
    eps_prime: float = 0.1       # ← unified with clf TAU_BINARY and reg TAU_SIGN
    epsilon_gap: float = 1e-3    # ← unified with clf/reg EPSILON_GAP
    tol: float = 1e-12
    max_quad_nodes: int = -1
    max_lin_edges_per_constr: int = -1

    # Dataset limits
    max_items_train: int = -1
    max_items_test: int = -1

    # Optional filters
    train_size_tag: Optional[str] = None
    test_size_tag: Optional[str] = None
    train_n: Optional[int] = None
    train_m_total: Optional[int] = None
    test_n: Optional[int] = None
    test_m_total: Optional[int] = None

    # Debug
    verbose: bool = True
    allow_row_order_fallback: bool = True


# ============================================================
# Helpers
# ============================================================

def _ts() -> str:
    return datetime.now().strftime("%H:%M:%S")


def _run_tag(cfg: RunConfig) -> str:
    def stem(p): return Path(p).stem if p else "none"
    tr = stem(cfg.train_pkl)
    te = stem(cfg.test_pkl) if cfg.test_pkl else "holdout"
    return f"gnn__train[{tr}]__test[{te}]__h{cfg.hidden}l{cfg.layers}ep{cfg.epochs}"


def _make_run_dir(cfg: RunConfig) -> Path:
    root = Path(cfg.output_root)
    tag = _run_tag(cfg)
    run_dir = root / tag
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir, tag


# ============================================================
# 1) PKL loader
# ============================================================

class RobustUnpickler(pickle.Unpickler):
    _cache = {}
    def find_class(self, module, name):
        try:
            return super().find_class(module, name)
        except Exception:
            key = (module, name)
            if key not in self._cache:
                self._cache[key] = type(name, (object,), {})
            return self._cache[key]


def load_instances_pkl(pkl_path: str) -> List[Any]:
    with open(pkl_path, "rb") as f:
        data = RobustUnpickler(f).load()
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        return data["instances"] if "instances" in data else list(data.values())
    return list(data)


# ============================================================
# 2) Helpers
# ============================================================

def get_field(obj, name, default=None):
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)


def to_key(v) -> Optional[str]:
    if v is None:
        return None
    if isinstance(v, float) and np.isnan(v):
        return None
    return str(v)


def safe_filename(s: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", s)
    return s[:180]


def as_1d(x) -> np.ndarray:
    return np.asarray(x, dtype=float).reshape(-1)


def as_2d(x) -> np.ndarray:
    a = np.asarray(x, dtype=float)
    if a.ndim != 2:
        raise ValueError(f"Expected 2D array, got shape {a.shape}")
    return a


def symmetrize(A: np.ndarray) -> np.ndarray:
    return 0.5 * (A + A.T)


def compute_delta(z_lp: float, z_sdp: float, eps: float = 1e-3) -> float:
    # eps aligned with EPSILON_GAP in clf/reg (1e-3)
    return float((z_lp - z_sdp) / (abs(z_lp) + abs(z_sdp) + eps))


def make_labels_from_row(row: pd.Series, eps_prime: float = 0.1,
                         epsilon_gap: float = 1e-3) -> Tuple[int, float, int]:
    """
    Returns (y_direct, delta, choice_mask).

    y_direct : 1 = use LP, 0 = use SDP  (single label used for training & eval)
    delta    : normalised LP−SDP gap (0.0 if not comparable)
    choice_mask : 1 if both solvers returned finite optimal values (comparable)

    Unbounded SDP → y_direct=1 (LP wins by default).
    Both finite   → y_direct = 1 if delta > -eps_prime else 0.
    """
    lp_status  = str(row.get("lp_status",  "")).lower()
    sdp_status = str(row.get("sdp_status", "")).lower()
    z_lp  = row.get("lp_obj",  np.nan)
    z_sdp = row.get("sdp_obj", np.nan)

    lp_ok  = ("optimal" in lp_status)  and pd.notna(z_lp)
    sdp_ok = ("optimal" in sdp_status) and pd.notna(z_sdp)

    if lp_ok and sdp_ok:
        # Both finite — compute normalised gap
        delta    = compute_delta(float(z_lp), float(z_sdp), eps=epsilon_gap)
        y_direct = 1 if (delta > -eps_prime) else 0
        return int(y_direct), float(delta), 1          # comparable

    if lp_ok and not sdp_ok:
        # SDP unbounded / failed → LP wins
        return 1, 0.0, 0                                # not comparable

    # LP failed or both failed → SDP wins (or unknown; default SDP)
    return 0, 0.0, 0


def make_edge_tensors(src, dst, attr):
    if len(src) == 0:
        return torch.empty((2, 0), dtype=torch.long), torch.empty((0, 1), dtype=torch.float)
    ei = torch.tensor([src, dst], dtype=torch.long)
    ea = torch.tensor(attr, dtype=torch.float)
    if ea.dim() == 1:
        ea = ea.view(-1, 1)
    return ei, ea


# ============================================================
# 3) Term selection
# ============================================================

def upper_tri_coeffs_for_half_quadratic(A: np.ndarray) -> np.ndarray:
    n = A.shape[0]
    iu, ju = np.triu_indices(n)
    coeff = A[iu, ju].astype(float)
    coeff[iu == ju] *= 0.5
    return coeff


def triu_flat_index(iu, ju, n) -> np.ndarray:
    iu = iu.astype(np.int64); ju = ju.astype(np.int64)
    return (iu * n - (iu * (iu - 1)) // 2 + (ju - iu)).astype(np.int64)


def select_top_pairs_across_all_matrices(A_list, tol, max_quad_nodes):
    n = A_list[0].shape[0]
    iu_all, ju_all = np.triu_indices(n)
    score = np.zeros(len(iu_all), dtype=float)
    for Ak in A_list:
        score = np.maximum(score, np.abs(upper_tri_coeffs_for_half_quadratic(symmetrize(Ak))))
    keep = score > tol
    iu = iu_all[keep].astype(int); ju = ju_all[keep].astype(int)
    if len(iu) == 0:
        return np.array([0], dtype=int), np.array([0], dtype=int)
    if max_quad_nodes is not None and len(iu) > max_quad_nodes:
        sc = score[keep]
        idx = np.argpartition(sc, -max_quad_nodes)[-max_quad_nodes:]
        order = np.argsort(-sc[idx])
        iu = iu[idx][order]; ju = ju[idx][order]
    return iu, ju


# ============================================================
# 4) Graph config
# ============================================================

@dataclass
class GraphConfig:
    tol: float = 1e-12
    eps_prime: float = 0.1       # unified threshold
    epsilon_gap: float = 1e-3    # unified epsilon
    max_quad_nodes: Optional[int] = None
    max_lin_edges_per_constr: Optional[int] = None


# ============================================================
# 5) Graph builder
# ============================================================

def build_tripartite_term_graph(inst, row: pd.Series, cfg: GraphConfig) -> HeteroData:
    A_list = get_field(inst, "A_list"); b_list = get_field(inst, "b_list")
    c_list = get_field(inst, "c_list"); l = get_field(inst, "l"); u = get_field(inst, "u")
    if any(x is None for x in [A_list, b_list, c_list, l, u]):
        raise ValueError("Instance missing required fields: A_list, b_list, c_list, l, u")

    A_list = [as_2d(Ak) for Ak in A_list]
    b_list = [as_1d(bk) for bk in b_list]
    c_list = [float(np.asarray(ck).reshape(-1)[0]) for ck in c_list]
    l = as_1d(l); u = as_1d(u)
    n = int(l.shape[0]); m_total = len(A_list); m_constr = m_total - 1
    data = HeteroData()

    p0 = b_list[0].astype(float)
    data["var"].x = torch.tensor(np.stack([p0, l, u, (u - l), np.ones_like(l)], axis=1), dtype=torch.float)

    cc = np.array([c_list[k] for k in range(1, m_total)], dtype=float)
    data["constr"].x = torch.tensor(np.stack([cc, np.ones_like(cc)], axis=1), dtype=torch.float)

    iu_sel, ju_sel = select_top_pairs_across_all_matrices(A_list, cfg.tol, cfg.max_quad_nodes)
    sel_idx = triu_flat_index(iu_sel, ju_sel, n)
    coeff0_sel = upper_tri_coeffs_for_half_quadratic(symmetrize(A_list[0]))[sel_idx]
    is_diag = (iu_sel == ju_sel).astype(float)
    quad_x = np.stack([coeff0_sel, is_diag, iu_sel / max(1.0, n-1), ju_sel / max(1.0, n-1), np.ones_like(is_diag)], axis=1)
    data["quad"].x = torch.tensor(quad_x, dtype=torch.float)

    vq_src, vq_dst, vq_attr = [], [], []
    for q_id, (i, j) in enumerate(zip(iu_sel.tolist(), ju_sel.tolist())):
        vq_src.append(int(i)); vq_dst.append(int(q_id)); vq_attr.append([1.0])
        if int(j) != int(i):
            vq_src.append(int(j)); vq_dst.append(int(q_id)); vq_attr.append([1.0])
    ei, ea = make_edge_tensors(vq_src, vq_dst, vq_attr)
    data["var", "inc", "quad"].edge_index = ei; data["var", "inc", "quad"].edge_attr = ea
    rev = torch.stack([ei[1], ei[0]], 0) if ei.numel() > 0 else torch.empty((2,0), dtype=torch.long)
    data["quad", "rev_inc", "var"].edge_index = rev; data["quad", "rev_inc", "var"].edge_attr = ea

    qc_src, qc_dst, qc_attr = [], [], []
    for k in range(1, m_total):
        coeffk_sel = upper_tri_coeffs_for_half_quadratic(symmetrize(A_list[k]))[sel_idx]
        for q_id in np.where(np.abs(coeffk_sel) > cfg.tol)[0]:
            qc_src.append(int(q_id)); qc_dst.append(int(k-1)); qc_attr.append([float(coeffk_sel[q_id])])
    ei, ea = make_edge_tensors(qc_src, qc_dst, qc_attr)
    data["quad", "qcoef", "constr"].edge_index = ei; data["quad", "qcoef", "constr"].edge_attr = ea
    rev = torch.stack([ei[1], ei[0]], 0) if ei.numel() > 0 else torch.empty((2,0), dtype=torch.long)
    data["constr", "rev_qcoef", "quad"].edge_index = rev; data["constr", "rev_qcoef", "quad"].edge_attr = ea

    vc_src, vc_dst, vc_attr = [], [], []
    for k in range(1, m_total):
        bk = b_list[k].astype(float)
        keep = np.arange(n) if (cfg.max_lin_edges_per_constr is None or cfg.max_lin_edges_per_constr >= n) \
               else np.argpartition(np.abs(bk), -cfg.max_lin_edges_per_constr)[-cfg.max_lin_edges_per_constr:]
        for i in keep:
            val = float(bk[int(i)])
            if abs(val) > cfg.tol:
                vc_src.append(int(i)); vc_dst.append(int(k-1)); vc_attr.append([val])
    ei, ea = make_edge_tensors(vc_src, vc_dst, vc_attr)
    data["var", "lin", "constr"].edge_index = ei; data["var", "lin", "constr"].edge_attr = ea
    rev = torch.stack([ei[1], ei[0]], 0) if ei.numel() > 0 else torch.empty((2,0), dtype=torch.long)
    data["constr", "rev_lin", "var"].edge_index = rev; data["constr", "rev_lin", "var"].edge_attr = ea

    data.global_x = torch.tensor([[float(c_list[0]), float(n)/500.0, float(m_constr)/500.0]], dtype=torch.float)

    # Single direct label: 1=LP, 0=SDP
    y_direct, delta, choice_mask = make_labels_from_row(
        row, eps_prime=cfg.eps_prime, epsilon_gap=cfg.epsilon_gap)
    data.y_direct    = torch.tensor([int(y_direct)],    dtype=torch.float)
    data.delta       = torch.tensor([float(delta)],     dtype=torch.float)
    data.choice_mask = torch.tensor([float(choice_mask)], dtype=torch.float)  # 1 if comparable
    data.row_key = to_key(row.get("id_global", row.get("id", row.get("__match_key__", "NA"))))
    return data


# ============================================================
# 6) Dataset
# ============================================================

def build_dataframe_from_instance_results(instances) -> pd.DataFrame:
    rows = []
    for inst in instances:
        res = get_field(inst, "results", {}) or {}
        if not isinstance(res, dict): res = {}
        rows.append({
            "instance_id": to_key(get_field(inst, "instance_id")),
            "group_id":    to_key(get_field(inst, "group_id")),
            "size_tag":    to_key(get_field(inst, "size_tag")),
            "id":          to_key(get_field(inst, "instance_id")),
            "lp_obj":      res.get("lp_obj", np.nan),
            "lp_status":   res.get("lp_status", ""),
            "sdp_obj":     res.get("sdp_obj", np.nan),
            "sdp_status":  res.get("sdp_status", ""),
        })
    df = pd.DataFrame(rows)
    if df[["lp_obj", "sdp_obj"]].notna().sum().sum() == 0 and (df["lp_status"].astype(str) == "").all():
        raise RuntimeError("No CSV provided and instances[].results has no LP/SDP labels.")
    return df


class QCQPTripartiteDataset(torch.utils.data.Dataset):
    def __init__(self, pkl_path, csv_path, cfg: GraphConfig,
                 cache_dir=None, max_items=None, require_labels=True,
                 filter_size_tag=None, filter_n=None, filter_m_total=None,
                 verbose=True, allow_row_order_fallback=True):
        super().__init__()
        self.cfg = cfg; self.cache_dir = cache_dir; self.verbose = verbose
        if cache_dir: os.makedirs(cache_dir, exist_ok=True)
        self.instances = load_instances_pkl(pkl_path)

        if csv_path and os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
        elif require_labels:
            df = build_dataframe_from_instance_results(self.instances)
        else:
            df = pd.DataFrame()

        self.candidate_keys = ["id_global", "id", "instance_id", "group_id", "size_tag"]
        self.instances_by_key: Dict[str, Any] = {}
        for idx, inst in enumerate(self.instances):
            for kn in self.candidate_keys:
                k = to_key(get_field(inst, kn))
                if k is not None: self.instances_by_key[k] = inst
            self.instances_by_key[f"__rowidx__{idx}"] = inst

        if df.shape[0] == 0:
            df = pd.DataFrame([{"__match_key__": f"__rowidx__{i}",
                "lp_obj": np.nan, "lp_status": "", "sdp_obj": np.nan, "sdp_status": ""}
                for i in range(len(self.instances))])

        def row_key_match(row):
            for k in self.candidate_keys:
                v = to_key(row.get(k))
                if v is not None and v in self.instances_by_key: return v
            return None

        match = [row_key_match(row) for _, row in df.iterrows()]
        keep_mask = [m is not None for m in match]
        matched_count = int(np.sum(keep_mask))

        if matched_count == 0 and allow_row_order_fallback:
            if len(df) == len(self.instances):
                if self.verbose: print(f"[{_ts()}] [match] No key matches; using ROW-ORDER fallback.")
                df = df.copy()
                df["__match_key__"] = [f"__rowidx__{i}" for i in range(len(df))]
            else:
                raise RuntimeError(
                    f"No rows matched and row-order fallback not possible "
                    f"(df={len(df)}, instances={len(self.instances)}).")
        else:
            df = df[np.array(keep_mask, dtype=bool)].copy()
            df["__match_key__"] = [m for m in match if m is not None]
            df.reset_index(drop=True, inplace=True)
            if self.verbose: print(f"[{_ts()}] [match] matched rows: {len(df)}/{len(match)}")

        if filter_size_tag is not None:
            df = df[df.get("size_tag", "").astype(str) == str(filter_size_tag)].copy()
            df.reset_index(drop=True, inplace=True)

        if filter_n is not None or filter_m_total is not None:
            ok_rows = []
            for i in range(len(df)):
                inst = self.instances_by_key[str(df.loc[i, "__match_key__"])]
                n_i = int(get_field(inst, "n", -1)); m_i = int(get_field(inst, "m", -1))
                ok = (filter_n is None or n_i == filter_n) and (filter_m_total is None or m_i == filter_m_total)
                ok_rows.append(ok)
            df = df[np.array(ok_rows, dtype=bool)].copy()
            df.reset_index(drop=True, inplace=True)

        yd_list, dd_list, mm_list = [], [], []
        for _, row in df.iterrows():
            yd, d, m = make_labels_from_row(row, eps_prime=self.cfg.eps_prime,
                                             epsilon_gap=self.cfg.epsilon_gap)
            yd_list.append(yd); dd_list.append(d); mm_list.append(m)
        df["__y_direct__"]    = yd_list
        df["__delta__"]       = dd_list
        df["__choice_mask__"] = mm_list

        if max_items and max_items > 0:
            df = df.iloc[:max_items].copy(); df.reset_index(drop=True, inplace=True)
        self.df = df

    def __len__(self): return len(self.df)

    def _cache_path(self, idx, row):
        return os.path.join(self.cache_dir, f"graph_{idx:08d}_{safe_filename(str(row['__match_key__']))}.pt")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if self.cache_dir:
            cp = self._cache_path(idx, row)
            if os.path.exists(cp): return torch.load(cp)
        inst = self.instances_by_key[str(row["__match_key__"])]
        g = build_tripartite_term_graph(inst, row, self.cfg)
        if self.cache_dir: torch.save(g, cp)
        return g


# ============================================================
# 7) Model  (single direct-decision head + delta regression head)
# ============================================================

class MLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, out_dim), nn.ReLU(), nn.Linear(out_dim, out_dim))
    def forward(self, x): return self.net(x)


def edge_weighted_sum(x_src, edge_index, edge_attr, num_dst):
    if edge_index.numel() == 0:
        return x_src.new_zeros((num_dst, x_src.size(1)))
    msg = x_src[edge_index[0]] * edge_attr
    return scatter(msg, edge_index[1], dim=0, dim_size=num_dst, reduce="sum")


class TripartiteTermGNN(nn.Module):
    """
    Single-head GNN: directly predicts LP(1) vs SDP(0).
    No bound/unboundedness head. Unbounded-SDP instances are labelled LP=1
    at the data level, so the model learns them naturally.
    """
    def __init__(self, hidden=64, layers=3, dropout=0.1):
        super().__init__()
        self.hidden = hidden; self.layers = layers; self.dropout = dropout
        H = hidden
        self.lin_var  = nn.Linear(5, H)
        self.lin_quad = nn.Linear(5, H)
        self.lin_con  = nn.Linear(2, H)
        self.f_v_to_q = nn.ModuleList([nn.Linear(H,H) for _ in range(layers)])
        self.f_v_to_c = nn.ModuleList([nn.Linear(H,H) for _ in range(layers)])
        self.f_q_to_c = nn.ModuleList([nn.Linear(H,H) for _ in range(layers)])
        self.f_c_to_q = nn.ModuleList([nn.Linear(H,H) for _ in range(layers)])
        self.f_c_to_v = nn.ModuleList([nn.Linear(H,H) for _ in range(layers)])
        self.f_q_to_v = nn.ModuleList([nn.Linear(H,H) for _ in range(layers)])
        self.g_q1 = nn.ModuleList([MLP(H*2,H) for _ in range(layers)])
        self.g_c  = nn.ModuleList([MLP(H*3,H) for _ in range(layers)])
        self.g_q2 = nn.ModuleList([MLP(H*2,H) for _ in range(layers)])
        self.g_v  = nn.ModuleList([MLP(H*3,H) for _ in range(layers)])
        head_in = H*3 + 3
        def _head():
            return nn.Sequential(nn.Linear(head_in,H), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(H,1))
        self.head_direct = _head()   # LP(1) vs SDP(0) — all instances
        self.head_delta  = _head()   # delta regression — comparable only

    def forward(self, data):
        h_v = self.lin_var(data["var"].x)
        h_q = self.lin_quad(data["quad"].x)
        h_c = self.lin_con(data["constr"].x)
        e_vq,w_vq = data["var","inc","quad"].edge_index,         data["var","inc","quad"].edge_attr
        e_vc,w_vc = data["var","lin","constr"].edge_index,        data["var","lin","constr"].edge_attr
        e_qc,w_qc = data["quad","qcoef","constr"].edge_index,     data["quad","qcoef","constr"].edge_attr
        e_cq,w_cq = data["constr","rev_qcoef","quad"].edge_index, data["constr","rev_qcoef","quad"].edge_attr
        e_cv,w_cv = data["constr","rev_lin","var"].edge_index,    data["constr","rev_lin","var"].edge_attr
        e_qv,w_qv = data["quad","rev_inc","var"].edge_index,      data["quad","rev_inc","var"].edge_attr
        nv = data["var"].num_nodes; nq = data["quad"].num_nodes; nc = data["constr"].num_nodes

        for t in range(self.layers):
            h_q_bar = self.g_q1[t](torch.cat([h_q, edge_weighted_sum(self.f_v_to_q[t](h_v), e_vq, w_vq, nq)], 1))
            h_c_new = self.g_c[t](torch.cat([h_c,
                                              edge_weighted_sum(self.f_v_to_c[t](h_v),   e_vc, w_vc, nc),
                                              edge_weighted_sum(self.f_q_to_c[t](h_q_bar), e_qc, w_qc, nc)], 1))
            h_q_new = self.g_q2[t](torch.cat([h_q_bar,
                                               edge_weighted_sum(self.f_c_to_q[t](h_c_new), e_cq, w_cq, nq)], 1))
            h_v_new = self.g_v[t](torch.cat([h_v,
                                              edge_weighted_sum(self.f_c_to_v[t](h_c_new), e_cv, w_cv, nv),
                                              edge_weighted_sum(self.f_q_to_v[t](h_q_new), e_qv, w_qv, nv)], 1))
            h_v = F.dropout(F.relu(h_v_new), p=self.dropout, training=self.training)
            h_q = F.dropout(F.relu(h_q_new), p=self.dropout, training=self.training)
            h_c = F.dropout(F.relu(h_c_new), p=self.dropout, training=self.training)

        B = int(data["var"].batch.max().item()) + 1
        pv = global_mean_pool(h_v, data["var"].batch, size=B)
        pq = global_mean_pool(h_q, data["quad"].batch, size=B) if nq > 0 else h_v.new_zeros((B, self.hidden))
        pc = global_mean_pool(h_c, data["constr"].batch, size=B) if nc > 0 else h_v.new_zeros((B, self.hidden))
        gx = data.global_x
        if gx.dim() == 1: gx = gx.view(-1, 3)
        if gx.size(0) != B: gx = gx.view(B, -1)
        g = torch.cat([pv, pq, pc, gx], dim=1)

        ld = self.head_direct(g).view(-1)   # direct LP/SDP logit
        lk = self.head_delta(g).view(-1)    # delta prediction
        return ld, lk


# ============================================================
# 8) Train / Eval
# ============================================================

def compute_pos_weight_binary(y, device):
    pos = float((y == 1).sum()); neg = float((y == 0).sum())
    if pos <= 0: return None
    return torch.tensor([neg / (pos + 1e-12)], dtype=torch.float, device=device)


def train_one_epoch(model, loader, opt, device, pw_direct, lam_d=0.25):
    model.train(); total = 0.0
    bce = (nn.BCEWithLogitsLoss(pos_weight=pw_direct).to(device)
           if pw_direct is not None else nn.BCEWithLogitsLoss().to(device))
    for batch in loader:
        batch = batch.to(device); opt.zero_grad()
        ld, lk = model(batch)
        yd = batch.y_direct.view(-1).float()
        d  = batch.delta.view(-1).float()
        m  = batch.choice_mask.view(-1).float()

        # Direct LP/SDP classification on ALL instances
        loss_direct = bce(ld, yd)
        # Delta regression on COMPARABLE instances only
        loss_delta = (((lk - d)**2 * m).sum() / (m.sum() + 1e-12)
                      if m.sum() > 0 else lk.new_tensor(0.0))
        loss = loss_direct + lam_d * loss_delta

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += float(loss.item()) * yd.numel()
    return total / max(1, len(loader.dataset))


def _safe_bal_acc(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(balanced_accuracy_score(y_true, y_pred))


def _safe_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(f1_score(y_true, y_pred, average="macro", zero_division=0))


@torch.no_grad()
def evaluate_full(model, loader, device):
    """Returns metrics dict, preds dict, and raw arrays for confusion matrix."""
    model.eval()
    yd_all, pd_all, m_all, d_all, dk_all, keys_all = [], [], [], [], [], []

    for batch in loader:
        batch = batch.to(device)
        ld, lk = model(batch)
        yd = batch.y_direct.view(-1).float()
        m  = batch.choice_mask.view(-1).float()

        yd_all.append(yd.cpu())
        pd_all.append((torch.sigmoid(ld) >= 0.5).float().cpu())
        m_all.append(m.cpu())
        d_all.append(batch.delta.view(-1).cpu())
        dk_all.append(lk.view(-1).cpu())

        if hasattr(batch, "row_key"):
            rk = batch.row_key
            if isinstance(rk, (list, tuple)):
                keys_all.extend([str(k) for k in rk])
            else:
                keys_all.extend([str(rk)] * int(yd.numel()))
        else:
            keys_all.extend(["NA"] * int(yd.numel()))

    yd = torch.cat(yd_all).numpy()
    pd_ = torch.cat(pd_all).numpy()
    m   = torch.cat(m_all).numpy()
    d   = torch.cat(d_all).numpy()
    dk  = torch.cat(dk_all).numpy()

    acc     = float((yd == pd_).mean())
    bal_acc = _safe_bal_acc(yd.astype(int), pd_.astype(int))
    f1      = _safe_f1(yd.astype(int), pd_.astype(int))
    mse     = float(((dk - d)**2 * m).sum() / (m.sum() + 1e-12)) if m.sum() > 0 else float("nan")

    metrics = {
        "acc": acc,
        "bal_acc": bal_acc,
        "f1_macro": f1,
        "delta_mse_comparable": mse,
        "coverage_comparable": float(m.mean()),
    }

    preds = {
        "row_key":     keys_all,
        "y_true":      yd.tolist(),
        "y_pred":      pd_.tolist(),
        "choice_mask": m.tolist(),
        "delta_true":  d.tolist(),
        "delta_pred":  dk.tolist(),
    }
    return metrics, preds, yd, pd_


# ============================================================
# 9) Deterministic split  (same hash logic as clf/reg)
# ============================================================

def make_deterministic_splits(keys: pd.Series, train: float = 0.8, val: float = 0.1):
    """
    Hash-based deterministic split.
    Bucket = hash(key) % 100.
    train  : bucket < train*100
    val    : train*100 <= bucket < (train+val)*100
    test   : bucket >= (train+val)*100

    With train=0.8, val=0.1:
      train = bucket <  80   (80%)
      val   = bucket 80–89  (10%)   ← internal monitoring only
      test  = bucket >= 90  (10%)   ← raw from this function

    In holdout mode, the REPORTED test set is bucket >= 80 (val ∪ test10 = 20%),
    matching clf/reg exactly. Val instances are included in the final test
    evaluation; this is valid because val is never used for model selection.
    """
    h = pd.util.hash_pandas_object(keys.astype(str), index=False).to_numpy(dtype=np.uint64)
    b = (h % 100).astype(np.int64)
    tc = int(train * 100)
    vc = int((train + val) * 100)
    return (np.where(b < tc)[0],
            np.where((b >= tc) & (b < vc))[0],
            np.where(b >= vc)[0])


def pick_split_key(df: pd.DataFrame) -> pd.Series:
    for col in ["id_global", "id", "instance_id", "group_id", "size_tag", "__match_key__"]:
        if col in df.columns and df[col].notna().any():
            return df[col].astype(str)
    return pd.Series(np.arange(len(df)), index=df.index).astype(str)


# ============================================================
# 10) Saving helpers
# ============================================================

def save_confusion_matrix(y_true, y_pred, labels, run_dir: Path, fname: str, title: str):
    cm = confusion_matrix(y_true.astype(int), y_pred.astype(int), labels=list(range(len(labels))))
    disp = ConfusionMatrixDisplay(cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(7, 5))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(run_dir / fname, dpi=200)
    plt.close()
    cm_df = pd.DataFrame(cm,
        index=[f"true_{l}" for l in labels],
        columns=[f"pred_{l}" for l in labels])
    cm_df.to_csv(run_dir / fname.replace(".png", ".csv"), index=True)
    print(f"[{_ts()}] [save] {fname}")


def save_epoch_log(epoch_log: List[dict], run_dir: Path, tag: str):
    pd.DataFrame(epoch_log).to_csv(run_dir / f"epoch_log_{tag}.csv", index=False)
    print(f"[{_ts()}] [save] epoch_log_{tag}.csv")


def save_results(metrics: dict, run_dir: Path, tag: str, split: str, cfg: RunConfig):
    row = {"split": split, "run_tag": tag, **metrics,
           "train_pkl": cfg.train_pkl, "test_pkl": cfg.test_pkl or "holdout",
           "epochs": cfg.epochs, "hidden": cfg.hidden, "layers": cfg.layers,
           "dropout": cfg.dropout, "lr": cfg.lr,
           "eps_prime": cfg.eps_prime, "epsilon_gap": cfg.epsilon_gap}
    pd.DataFrame([row]).to_csv(run_dir / f"results_gnn_{tag}.csv", index=False)
    print(f"[{_ts()}] [save] results_gnn_{tag}.csv")


def save_predictions(preds: dict, run_dir: Path, tag: str, split: str):
    df = pd.DataFrame(preds)
    df.insert(0, "split", split)
    df.to_csv(run_dir / f"preds_{tag}.csv", index=False)
    print(f"[{_ts()}] [save] preds_{tag}.csv")


def save_config(cfg: RunConfig, run_dir: Path, tag: str):
    pd.DataFrame([asdict(cfg)]).to_csv(run_dir / f"config_{tag}.csv", index=False)
    print(f"[{_ts()}] [save] config_{tag}.csv")


# ============================================================
# 11) main()
# ============================================================

def main(run_cfg: Optional[RunConfig] = None):
    """
    Notebook usage:
        cfg = RunConfig(train_pkl="...", train_csv="...", epochs=20)
        model, metrics = main(cfg)
    """
    if run_cfg is None:
        run_cfg = RunConfig()

    np.random.seed(0); torch.manual_seed(0)

    run_dir, tag = _make_run_dir(run_cfg)
    print(f"[{_ts()}] ══════════════════════════════════════════")
    print(f"[{_ts()}] GNN PIPELINE  tag={tag}")
    print(f"[{_ts()}] Output dir: {run_dir.resolve()}")
    print(f"[{_ts()}] eps_prime={run_cfg.eps_prime}  epsilon_gap={run_cfg.epsilon_gap}")
    print(f"[{_ts()}] ══════════════════════════════════════════")

    gcfg = GraphConfig(
        tol=run_cfg.tol,
        eps_prime=run_cfg.eps_prime,
        epsilon_gap=run_cfg.epsilon_gap,
        max_quad_nodes=None if run_cfg.max_quad_nodes <= 0 else int(run_cfg.max_quad_nodes),
        max_lin_edges_per_constr=None if run_cfg.max_lin_edges_per_constr <= 0 else int(run_cfg.max_lin_edges_per_constr),
    )
    mit = None if run_cfg.max_items_train <= 0 else int(run_cfg.max_items_train)
    mie = None if run_cfg.max_items_test  <= 0 else int(run_cfg.max_items_test)

    train_ds_full = QCQPTripartiteDataset(
        run_cfg.train_pkl, run_cfg.train_csv, gcfg,
        cache_dir=run_cfg.cache_dir, max_items=mit, require_labels=True,
        filter_size_tag=run_cfg.train_size_tag, filter_n=run_cfg.train_n,
        filter_m_total=run_cfg.train_m_total,
        verbose=run_cfg.verbose, allow_row_order_fallback=run_cfg.allow_row_order_fallback)

    if len(train_ds_full) == 0:
        raise RuntimeError("Train dataset empty. Check PKL/CSV and keys.")

    if run_cfg.test_pkl:
        # ── External test file provided ──────────────────────────────────────
        test_ds = QCQPTripartiteDataset(
            run_cfg.test_pkl, run_cfg.test_csv, gcfg,
            cache_dir=None, max_items=mie, require_labels=True,
            filter_size_tag=run_cfg.test_size_tag, filter_n=run_cfg.test_n,
            filter_m_total=run_cfg.test_m_total,
            verbose=run_cfg.verbose, allow_row_order_fallback=run_cfg.allow_row_order_fallback)
        if len(test_ds) == 0:
            raise RuntimeError("Test dataset empty. Check PKL/CSV and keys.")
        # Split train into 90% train / 10% val (test is external)
        tr_idx, va_idx, _ = make_deterministic_splits(pick_split_key(train_ds_full.df), 0.9, 0.1)
        test_split_name = "external_test"
    else:
        # ── Holdout split ─────────────────────────────────────────────────────
        # train : bucket <  80  (80%)   — used for training
        # val   : bucket 80–89 (10%)   — used for per-epoch monitoring only
        # test  : bucket >= 80 (20%)   — val ∪ test10; reported as final metrics
        #
        # clf/reg test = bucket >= 80  →  IDENTICAL 20% test set across all three models.
        # Val instances (80–89) are re-used in the test evaluation: this is fine
        # because val is never used to select or modify the model (no early stopping).
        tr_idx, va_idx, te_idx = make_deterministic_splits(pick_split_key(train_ds_full.df), 0.8, 0.1)
        # test = val ∪ test10  (bucket >= 80)
        test_idx_20 = np.concatenate([va_idx, te_idx])
        test_ds = Subset(train_ds_full, test_idx_20.tolist())
        test_split_name = "holdout_test"

    train_ds = Subset(train_ds_full, tr_idx.tolist())
    val_ds   = Subset(train_ds_full, va_idx.tolist())
    print(f"[{_ts()}] [data] train={len(train_ds)} | val={len(val_ds)} | test={len(test_ds)}", flush=True)

    train_loader = DataLoader(train_ds, batch_size=run_cfg.batch_train, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=run_cfg.batch_eval,  shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=run_cfg.batch_eval,  shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[{_ts()}] [device] {device}", flush=True)

    df_tr    = train_ds_full.df.iloc[tr_idx]
    pw_direct = compute_pos_weight_binary(df_tr["__y_direct__"].to_numpy(float), device)
    print(f"[{_ts()}] [weights] pw_direct={None if pw_direct is None else round(float(pw_direct), 4)}")

    model = TripartiteTermGNN(run_cfg.hidden, run_cfg.layers, run_cfg.dropout).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=run_cfg.lr, weight_decay=run_cfg.weight_decay)

    # ── Training loop ──────────────────────────────────────
    epoch_log = []
    for epoch in range(1, run_cfg.epochs + 1):
        loss = train_one_epoch(model, train_loader, opt, device, pw_direct, run_cfg.lambda_delta)
        vm, _, _, _ = evaluate_full(model, val_loader, device)
        row = {"epoch": epoch, "train_loss": loss, **{f"val_{k}": v for k, v in vm.items()}}
        epoch_log.append(row)
        print(f"[{_ts()}] Epoch {epoch:02d} | loss={loss:.4f} | "
              f"acc={vm['acc']:.4f} | bal_acc={vm['bal_acc']:.4f} | "
              f"f1_macro={vm['f1_macro']:.4f} | delta_mse={vm['delta_mse_comparable']:.6f} | "
              f"cov={vm['coverage_comparable']:.3f}", flush=True)

    save_epoch_log(epoch_log, run_dir, tag)

    # ── Test evaluation ────────────────────────────────────
    test_metrics, preds, yd, pd_ = evaluate_full(model, test_loader, device)
    print(f"\n[{_ts()}] TEST: {test_metrics}", flush=True)

    save_predictions(preds, run_dir, tag, test_split_name)

    save_confusion_matrix(yd, pd_, ["SDP(0)", "LP(1)"],
        run_dir, f"cm_{tag}.png", "Direct LP/SDP Prediction")

    save_results(test_metrics, run_dir, tag, test_split_name, run_cfg)
    save_config(run_cfg, run_dir, tag)

    # ── Graph sanity ───────────────────────────────────────
    g0 = train_ds_full[0]
    print(f"\n[{_ts()}] [graph sanity]")
    print(f"  var    : {g0['var'].num_nodes} nodes  shape={list(g0['var'].x.shape)}")
    print(f"  quad   : {g0['quad'].num_nodes} nodes  shape={list(g0['quad'].x.shape)}")
    print(f"  constr : {g0['constr'].num_nodes} nodes  shape={list(g0['constr'].x.shape)}")
    for et in g0.edge_types:
        print(f"  {et}: E={g0[et].edge_index.shape[1]}  attr={list(g0[et].edge_attr.shape)}")

    written = sorted(p.name for p in run_dir.glob("*"))
    print(f"\n[{_ts()}] Files written ({len(written)}): {written}")

    return model, test_metrics

In [29]:
cfg = RunConfig(
    train_pkl="instances_5_2_50.pkl",
    train_csv="instancesobj_5_2_50.csv",
    test_pkl="instances_5_10_10.pkl",
    test_csv="instancesobj_5_10_10.csv",
    epochs=10,
)
model, metrics = main(cfg)

[10:09:22] ══════════════════════════════════════════
[10:09:22] GNN PIPELINE  tag=gnn__train[instances_5_2_50]__test[instances_5_10_10]__h64l3ep10
[10:09:22] Output dir: C:\Users\suuser\Desktop\minlplib\gnn_outputs\gnn__train[instances_5_2_50]__test[instances_5_10_10]__h64l3ep10
[10:09:22] eps_prime=0.1  epsilon_gap=0.001
[10:09:22] ══════════════════════════════════════════
[10:09:26] [match] matched rows: 50000/50000
[10:09:29] [match] matched rows: 10000/10000
[10:09:30] [data] train=45062 | val=4938 | test=10000
[10:09:30] [device] cpu
[10:09:30] [weights] pw_direct=0.4429
[10:13:50] Epoch 01 | loss=0.9212 | acc=0.9433 | bal_acc=0.9112 | f1_macro=0.9298 | delta_mse=0.049259 | cov=0.542
[10:18:12] Epoch 02 | loss=0.2088 | acc=0.9350 | bal_acc=0.9258 | f1_macro=0.9237 | delta_mse=0.047277 | cov=0.542
[10:22:35] Epoch 03 | loss=0.1439 | acc=0.9293 | bal_acc=0.9172 | f1_macro=0.9167 | delta_mse=0.037364 | cov=0.542
[10:27:02] Epoch 04 | loss=0.1952 | acc=0.9360 | bal_acc=0.9192 | f1_m

In [ ]:
# qcqp_gnn_classifier_fixed.py
#
# FIXES APPLIED:
#
#   FIX 1 | COMPARABLE INSTANCE DEFINITION — aligned to ML pipelines.
#            Old: lp_ok requires _is_optimal(lp_status) AND finite AND not unb AND not excl
#                 (any row with a finite obj but status not in OPTIMAL_STATUSES was
#                  treated as non-comparable, defaulting to SDP)
#            New: lp_ok requires finite AND not unbounded AND not excluded
#                 (matches ML Classifier, ML Regressor, and GNN Regressor)
#
#   FIX 2 | SPLIT — holdout now uses 72/8/20 (train/val/test):
#            - test  = bucket >= 80  (same 20% instances as ML pipelines)
#            - val   = 72 <= bucket < 80  (~8%, = 10% of training portion, no leakage)
#            - train = bucket < 72
#            External test: unchanged (90% train / 10% val from training PKL).
#            Old holdout used 80/10/10 then concatenated val+test (buckets 80-89)
#            into the reported test set, causing val leakage into test reporting.

import os
import re
import pickle
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Subset

from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool
from torch_geometric.utils import scatter

from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    balanced_accuracy_score, f1_score,
)


# ============================================================
# 0) Status constants
# ============================================================

OPTIMAL_STATUSES = frozenset({
    "optimal",
    "optimal_inaccurate",
})

UNBOUNDED_STATUSES = frozenset({
    "unbounded",
    "unbounded_inaccurate",
    "unbounded_uncertified",
})

EXCLUDE_STATUSES = frozenset({
    "infeasible", "infeasible_inaccurate", "infeasible_or_unbounded",
    "primal_infeasible", "dual_infeasible_cert", "unknown",
    "numeric_error", "time_limit", "iteration_limit", "node_limit",
    "interrupted", "memory_limit",
})


# ============================================================
# 1) RunConfig
# ============================================================

@dataclass
class RunConfig:
    # Data
    train_pkl: str = "instances_5_10_10.pkl"
    train_csv: Optional[str] = "instancesobj_5_10_10.csv"
    test_pkl: Optional[str] = "instances_5_1_50.pkl"
    test_csv: Optional[str] = "instancesobj_5_1_50.csv"
    cache_dir: Optional[str] = None

    # Output
    output_root: str = "gnn_outputs"

    # Model
    epochs: int = 10
    hidden: int = 64
    layers: int = 3
    dropout: float = 0.1
    lr: float = 1e-3
    weight_decay: float = 1e-5
    lambda_delta: float = 0.25

    # Batching
    batch_train: int = 8
    batch_eval: int = 16

    # Unified with regression code
    epsilon_gap: float = 1e-3
    tau_sign: float = 0.1
    eps_prime: float = 0.1      # LP-bias threshold: LP if delta > -eps_prime

    # Sample weights
    weight_alpha: float = 0.30  # w = (1-alpha) + alpha*|delta|

    tol: float = 1e-12
    max_quad_nodes: int = -1
    max_lin_edges_per_constr: int = -1

    # Dataset limits
    max_items_train: int = -1
    max_items_test: int = -1

    # Optional filters
    train_size_tag: Optional[str] = None
    test_size_tag: Optional[str] = None
    train_n: Optional[int] = None
    train_m_total: Optional[int] = None
    test_n: Optional[int] = None
    test_m_total: Optional[int] = None

    # Debug
    verbose: bool = True
    allow_row_order_fallback: bool = True


# ============================================================
# Helpers
# ============================================================

def _ts() -> str:
    return datetime.now().strftime("%H:%M:%S")


def _run_tag(cfg: RunConfig) -> str:
    def stem(p): return Path(p).stem if p else "none"
    tr = stem(cfg.train_pkl)
    te = stem(cfg.test_pkl) if cfg.test_pkl else "holdout"
    return (f"gnnclf__train[{tr}]__test[{te}]"
            f"__h{cfg.hidden}l{cfg.layers}ep{cfg.epochs}__gap[fixed]")


def _make_run_dir(cfg: RunConfig) -> Tuple[Path, str]:
    root = Path(cfg.output_root)
    tag = _run_tag(cfg)
    run_dir = root / tag
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir, tag


# ============================================================
# 2) PKL loader
# ============================================================

class RobustUnpickler(pickle.Unpickler):
    _cache = {}
    def find_class(self, module, name):
        try:
            return super().find_class(module, name)
        except Exception:
            key = (module, name)
            if key not in self._cache:
                self._cache[key] = type(name, (object,), {})
            return self._cache[key]


def load_instances_pkl(pkl_path: str) -> List[Any]:
    with open(pkl_path, "rb") as f:
        data = RobustUnpickler(f).load()
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        return data["instances"] if "instances" in data else list(data.values())
    return list(data)


# ============================================================
# 3) Low-level helpers
# ============================================================

def get_field(obj, name, default=None):
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)


def to_key(v) -> Optional[str]:
    if v is None:
        return None
    if isinstance(v, float) and np.isnan(v):
        return None
    return str(v)


def safe_filename(s: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", s)
    return s[:180]


def as_1d(x) -> np.ndarray:
    return np.asarray(x, dtype=float).reshape(-1)


def as_2d(x) -> np.ndarray:
    a = np.asarray(x, dtype=float)
    if a.ndim != 2:
        raise ValueError(f"Expected 2D array, got shape {a.shape}")
    return a


def symmetrize(A: np.ndarray) -> np.ndarray:
    return 0.5 * (A + A.T)


_FEAT_CLIP = 1e4

def _safe(arr: np.ndarray) -> np.ndarray:
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(arr, -_FEAT_CLIP, _FEAT_CLIP)


def _safe_scalar(v: float) -> float:
    if not np.isfinite(v):
        return 0.0
    return float(np.clip(v, -_FEAT_CLIP, _FEAT_CLIP))


def make_edge_tensors(src, dst, attr):
    if len(src) == 0:
        return torch.empty((2, 0), dtype=torch.long), torch.empty((0, 1), dtype=torch.float)
    ei = torch.tensor([src, dst], dtype=torch.long)
    ea = torch.tensor(attr, dtype=torch.float)
    if ea.dim() == 1:
        ea = ea.view(-1, 1)
    return ei, ea


def _norm_status(s) -> str:
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    return str(s).strip().lower()


def _is_optimal(status: str) -> bool:
    return status in OPTIMAL_STATUSES


def _is_unbounded(status: str) -> bool:
    return status in UNBOUNDED_STATUSES


def _is_excluded(status: str) -> bool:
    return status in EXCLUDE_STATUSES


def make_edge_tensors_safe(src, dst, attr):
    return make_edge_tensors(src, dst, attr)


# ============================================================
# 4) Delta / labels  [FIX 1 — comparable definition aligned to ML pipelines]
# ============================================================

def compute_delta(z_lp: float, z_sdp: float, eps: float = 1e-3) -> float:
    """Fixed additive delta — unified across all four pipelines."""
    denom = abs(z_lp) + abs(z_sdp) + float(eps)
    return float((z_lp - z_sdp) / denom)


def make_labels_from_row(row: pd.Series,
                         eps_prime: float = 0.1,
                         epsilon_gap: float = 1e-3) -> Tuple[int, float, int]:
    """
    Returns (y_direct, delta, choice_mask).

    y_direct    : 1 = LP, 0 = SDP
    delta       : fixed-additive gap (0.0 if not comparable)
    choice_mask : 1 only when BOTH solvers returned finite results that are not
                  unbounded or excluded (comparable instances).

    Comparable definition [FIX 1 — aligned to ML pipelines]:
        lp_ok  = isfinite(z_lp) AND not unbounded AND not excluded
        sdp_ok = isfinite(z_sdp) AND not unbounded AND not excluded
        (OPTIMAL_STATUSES check removed — matches ML Classifier / ML Regressor)

    Decision rule (LP-biased):
        y_direct = 1 if delta > -eps_prime else 0

    Unbounded (choice_mask = 0, excluded from delta-MSE):
        SDP unbounded + LP ok  -> y=1, delta=0.0, mask=0
        LP unbounded + SDP ok  -> y=0, delta=0.0, mask=0
    """
    lp_status  = _norm_status(row.get("lp_status",  ""))
    sdp_status = _norm_status(row.get("sdp_status", ""))

    z_lp  = row.get("lp_obj",  np.nan)
    z_sdp = row.get("sdp_obj", np.nan)

    lp_unb  = _is_unbounded(lp_status)  or (pd.notna(z_lp)  and np.isneginf(float(z_lp)))
    sdp_unb = _is_unbounded(sdp_status) or (pd.notna(z_sdp) and np.isneginf(float(z_sdp)))

    lp_excl  = _is_excluded(lp_status)
    sdp_excl = _is_excluded(sdp_status)

    # [FIX 1] Comparable = finite AND not unbounded AND not excluded
    # (OPTIMAL_STATUSES check removed to match ML pipelines)
    lp_ok = (
        pd.notna(z_lp)
        and np.isfinite(float(z_lp))
        and not lp_unb
        and not lp_excl
    )
    sdp_ok = (
        pd.notna(z_sdp)
        and np.isfinite(float(z_sdp))
        and not sdp_unb
        and not sdp_excl
    )

    if lp_ok and sdp_ok:
        # Comparable: compute delta, LP-biased label
        delta = compute_delta(float(z_lp), float(z_sdp), eps=epsilon_gap)
        y_direct = 1 if delta > -eps_prime else 0
        return int(y_direct), float(delta), 1   # choice_mask = 1

    # Unbounded -> excluded from delta MSE (choice_mask = 0)
    if lp_ok and not sdp_ok:
        return 1, 0.0, 0

    # LP failed, or both failed -> SDP (default)
    return 0, 0.0, 0


# ============================================================
# 5) Term selection
# ============================================================

def upper_tri_coeffs_for_half_quadratic(A: np.ndarray) -> np.ndarray:
    n = A.shape[0]
    iu, ju = np.triu_indices(n)
    coeff = A[iu, ju].astype(float)
    coeff[iu == ju] *= 0.5
    return coeff


def triu_flat_index(iu, ju, n) -> np.ndarray:
    iu = iu.astype(np.int64); ju = ju.astype(np.int64)
    return (iu * n - (iu * (iu - 1)) // 2 + (ju - iu)).astype(np.int64)


def select_top_pairs_across_all_matrices(A_list, tol, max_quad_nodes):
    n = A_list[0].shape[0]
    iu_all, ju_all = np.triu_indices(n)
    score = np.zeros(len(iu_all), dtype=float)
    for Ak in A_list:
        score = np.maximum(score, np.abs(upper_tri_coeffs_for_half_quadratic(symmetrize(Ak))))
    keep = score > tol
    iu = iu_all[keep].astype(int); ju = ju_all[keep].astype(int)
    if len(iu) == 0:
        return np.array([0], dtype=int), np.array([0], dtype=int)
    if max_quad_nodes is not None and len(iu) > max_quad_nodes:
        sc = score[keep]
        idx = np.argpartition(sc, -max_quad_nodes)[-max_quad_nodes:]
        order = np.argsort(-sc[idx])
        iu = iu[idx][order]; ju = ju[idx][order]
    return iu, ju


# ============================================================
# 6) Graph config
# ============================================================

@dataclass
class GraphConfig:
    tol: float = 1e-12
    eps_prime: float = 0.1
    tau_sign: float = 0.1
    epsilon_gap: float = 1e-3
    max_quad_nodes: Optional[int] = None
    max_lin_edges_per_constr: Optional[int] = None


# ============================================================
# 7) Graph builder
# ============================================================

def build_tripartite_term_graph(inst, row: pd.Series, cfg: GraphConfig) -> HeteroData:
    A_list = get_field(inst, "A_list"); b_list = get_field(inst, "b_list")
    c_list = get_field(inst, "c_list"); l = get_field(inst, "l"); u = get_field(inst, "u")
    if any(x is None for x in [A_list, b_list, c_list, l, u]):
        raise ValueError("Instance missing required fields: A_list, b_list, c_list, l, u")

    A_list = [as_2d(Ak) for Ak in A_list]
    b_list = [as_1d(bk) for bk in b_list]
    c_list = [float(np.asarray(ck).reshape(-1)[0]) for ck in c_list]
    l = as_1d(l); u = as_1d(u)
    n = int(l.shape[0]); m_total = len(A_list); m_constr = m_total - 1
    data = HeteroData()

    p0 = _safe(b_list[0].astype(float))
    l_s = _safe(l); u_s = _safe(u)
    data["var"].x = torch.tensor(
        np.stack([p0, l_s, u_s, (u_s - l_s), np.ones_like(l_s)], axis=1), dtype=torch.float)

    cc = _safe(np.array([_safe_scalar(c_list[k]) for k in range(1, m_total)], dtype=float))
    data["constr"].x = torch.tensor(np.stack([cc, np.ones_like(cc)], axis=1), dtype=torch.float)

    iu_sel, ju_sel = select_top_pairs_across_all_matrices(A_list, cfg.tol, cfg.max_quad_nodes)
    sel_idx = triu_flat_index(iu_sel, ju_sel, n)
    coeff0_sel = _safe(upper_tri_coeffs_for_half_quadratic(symmetrize(A_list[0]))[sel_idx])
    is_diag = (iu_sel == ju_sel).astype(float)
    quad_x = np.stack([coeff0_sel, is_diag,
                       iu_sel / max(1.0, n - 1), ju_sel / max(1.0, n - 1),
                       np.ones_like(is_diag)], axis=1)
    data["quad"].x = torch.tensor(quad_x, dtype=torch.float)

    # var -> quad
    vq_src, vq_dst, vq_attr = [], [], []
    for q_id, (i, j) in enumerate(zip(iu_sel.tolist(), ju_sel.tolist())):
        vq_src.append(int(i)); vq_dst.append(int(q_id)); vq_attr.append([1.0])
        if int(j) != int(i):
            vq_src.append(int(j)); vq_dst.append(int(q_id)); vq_attr.append([1.0])
    ei, ea = make_edge_tensors(vq_src, vq_dst, vq_attr)
    data["var", "inc", "quad"].edge_index = ei
    data["var", "inc", "quad"].edge_attr = ea
    rev = torch.stack([ei[1], ei[0]], 0) if ei.numel() > 0 else torch.empty((2, 0), dtype=torch.long)
    data["quad", "rev_inc", "var"].edge_index = rev
    data["quad", "rev_inc", "var"].edge_attr = ea

    # quad -> constr
    qc_src, qc_dst, qc_attr = [], [], []
    for k in range(1, m_total):
        coeffk_sel = _safe(upper_tri_coeffs_for_half_quadratic(symmetrize(A_list[k]))[sel_idx])
        for q_id in np.where(np.abs(coeffk_sel) > cfg.tol)[0]:
            qc_src.append(int(q_id)); qc_dst.append(int(k - 1)); qc_attr.append([float(coeffk_sel[q_id])])
    ei, ea = make_edge_tensors(qc_src, qc_dst, qc_attr)
    data["quad", "qcoef", "constr"].edge_index = ei
    data["quad", "qcoef", "constr"].edge_attr = ea
    rev = torch.stack([ei[1], ei[0]], 0) if ei.numel() > 0 else torch.empty((2, 0), dtype=torch.long)
    data["constr", "rev_qcoef", "quad"].edge_index = rev
    data["constr", "rev_qcoef", "quad"].edge_attr = ea

    # var -> constr
    vc_src, vc_dst, vc_attr = [], [], []
    for k in range(1, m_total):
        bk = _safe(b_list[k].astype(float))
        keep = np.arange(n) if (cfg.max_lin_edges_per_constr is None or cfg.max_lin_edges_per_constr >= n) \
               else np.argpartition(np.abs(bk), -cfg.max_lin_edges_per_constr)[-cfg.max_lin_edges_per_constr:]
        for i in keep:
            val = float(bk[int(i)])
            if abs(val) > cfg.tol:
                vc_src.append(int(i)); vc_dst.append(int(k - 1)); vc_attr.append([val])
    ei, ea = make_edge_tensors(vc_src, vc_dst, vc_attr)
    data["var", "lin", "constr"].edge_index = ei
    data["var", "lin", "constr"].edge_attr = ea
    rev = torch.stack([ei[1], ei[0]], 0) if ei.numel() > 0 else torch.empty((2, 0), dtype=torch.long)
    data["constr", "rev_lin", "var"].edge_index = rev
    data["constr", "rev_lin", "var"].edge_attr = ea

    data.global_x = torch.tensor(
        [[_safe_scalar(c_list[0]), float(n) / 500.0, float(m_constr) / 500.0]],
        dtype=torch.float)

    y_direct, delta, choice_mask = make_labels_from_row(
        row, eps_prime=cfg.eps_prime, epsilon_gap=cfg.epsilon_gap)

    data.y_direct    = torch.tensor([int(y_direct)],      dtype=torch.float)
    data.delta       = torch.tensor([float(delta)],       dtype=torch.float)
    data.choice_mask = torch.tensor([float(choice_mask)], dtype=torch.float)
    data.row_key = to_key(row.get("id_global", row.get("id", row.get("__match_key__", "NA"))))
    return data


# ============================================================
# 8) Dataset
# ============================================================

def build_dataframe_from_instance_results(instances) -> pd.DataFrame:
    rows = []
    for inst in instances:
        res = get_field(inst, "results", {}) or {}
        if not isinstance(res, dict): res = {}
        rows.append({
            "instance_id": to_key(get_field(inst, "instance_id")),
            "group_id":    to_key(get_field(inst, "group_id")),
            "size_tag":    to_key(get_field(inst, "size_tag")),
            "id":          to_key(get_field(inst, "instance_id")),
            "lp_obj":      res.get("lp_obj", np.nan),
            "lp_status":   res.get("lp_status", ""),
            "sdp_obj":     res.get("sdp_obj", np.nan),
            "sdp_status":  res.get("sdp_status", ""),
        })
    df = pd.DataFrame(rows)
    if df[["lp_obj", "sdp_obj"]].notna().sum().sum() == 0 and (df["lp_status"].astype(str) == "").all():
        raise RuntimeError("No CSV provided and instances[].results has no LP/SDP labels.")
    return df


class QCQPTripartiteDataset(torch.utils.data.Dataset):
    def __init__(self, pkl_path, csv_path, cfg: GraphConfig,
                 cache_dir=None, max_items=None, require_labels=True,
                 filter_size_tag=None, filter_n=None, filter_m_total=None,
                 verbose=True, allow_row_order_fallback=True):
        super().__init__()
        self.cfg = cfg; self.cache_dir = cache_dir; self.verbose = verbose
        if cache_dir: os.makedirs(cache_dir, exist_ok=True)
        self.instances = load_instances_pkl(pkl_path)

        if csv_path and os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
        elif require_labels:
            df = build_dataframe_from_instance_results(self.instances)
        else:
            df = pd.DataFrame()

        self.candidate_keys = ["id_global", "id", "instance_id", "group_id", "size_tag"]
        self.instances_by_key: Dict[str, Any] = {}
        for idx, inst in enumerate(self.instances):
            for kn in self.candidate_keys:
                k = to_key(get_field(inst, kn))
                if k is not None: self.instances_by_key[k] = inst
            self.instances_by_key[f"__rowidx__{idx}"] = inst

        if df.shape[0] == 0:
            df = pd.DataFrame([{"__match_key__": f"__rowidx__{i}",
                                "lp_obj": np.nan, "lp_status": "",
                                "sdp_obj": np.nan, "sdp_status": ""}
                               for i in range(len(self.instances))])

        def row_key_match(row):
            for k in self.candidate_keys:
                v = to_key(row.get(k))
                if v is not None and v in self.instances_by_key:
                    return v
            return None

        match = [row_key_match(row) for _, row in df.iterrows()]
        keep_mask = [m is not None for m in match]
        matched_count = int(np.sum(keep_mask))

        if matched_count == 0 and allow_row_order_fallback:
            if len(df) == len(self.instances):
                if self.verbose:
                    print(f"[{_ts()}] [match] No key matches; using ROW-ORDER fallback.")
                df = df.copy()
                df["__match_key__"] = [f"__rowidx__{i}" for i in range(len(df))]
            else:
                raise RuntimeError(
                    f"No rows matched and row-order fallback not possible "
                    f"(df={len(df)}, instances={len(self.instances)}).")
        else:
            df = df[np.array(keep_mask, dtype=bool)].copy()
            df["__match_key__"] = [m for m in match if m is not None]
            df.reset_index(drop=True, inplace=True)
            if self.verbose:
                print(f"[{_ts()}] [match] matched rows: {len(df)}/{len(match)}")

        if filter_size_tag is not None:
            df = df[df.get("size_tag", "").astype(str) == str(filter_size_tag)].copy()
            df.reset_index(drop=True, inplace=True)

        if filter_n is not None or filter_m_total is not None:
            ok_rows = []
            for i in range(len(df)):
                inst = self.instances_by_key[str(df.loc[i, "__match_key__"])]
                n_i = int(get_field(inst, "n", -1)); m_i = int(get_field(inst, "m", -1))
                ok = (filter_n is None or n_i == filter_n) and (filter_m_total is None or m_i == filter_m_total)
                ok_rows.append(ok)
            df = df[np.array(ok_rows, dtype=bool)].copy()
            df.reset_index(drop=True, inplace=True)

        yd_list, dd_list, mm_list = [], [], []
        for _, row in df.iterrows():
            yd, d, m = make_labels_from_row(
                row, eps_prime=self.cfg.eps_prime, epsilon_gap=self.cfg.epsilon_gap)
            yd_list.append(yd); dd_list.append(d); mm_list.append(m)
        df["__y_direct__"]    = yd_list
        df["__delta__"]       = dd_list
        df["__choice_mask__"] = mm_list

        if max_items and max_items > 0:
            df = df.iloc[:max_items].copy(); df.reset_index(drop=True, inplace=True)
        self.df = df

    def __len__(self): return len(self.df)

    def _cache_path(self, idx, row):
        return os.path.join(
            self.cache_dir,
            f"graph_{idx:08d}_{safe_filename(str(row['__match_key__']))}.pt")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if self.cache_dir:
            cp = self._cache_path(idx, row)
            if os.path.exists(cp): return torch.load(cp)
        inst = self.instances_by_key[str(row["__match_key__"])]
        g = build_tripartite_term_graph(inst, row, self.cfg)
        if self.cache_dir: torch.save(g, cp)
        return g


# ============================================================
# 9) Model
# ============================================================

class MLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, out_dim), nn.ReLU(), nn.Linear(out_dim, out_dim))
    def forward(self, x): return self.net(x)


def edge_weighted_sum(x_src, edge_index, edge_attr, num_dst):
    if edge_index.numel() == 0:
        return x_src.new_zeros((num_dst, x_src.size(1)))
    msg = x_src[edge_index[0]] * edge_attr
    return scatter(msg, edge_index[1], dim=0, dim_size=num_dst, reduce="sum")


class TripartiteTermGNN(nn.Module):
    """
    Dual-head GNN classifier.
      head_direct : BCE  — predicts LP(1) vs SDP(0) on ALL instances.
      head_delta  : MSE  — predicts delta on COMPARABLE instances only (choice_mask=1).
    """
    def __init__(self, hidden=64, layers=3, dropout=0.1):
        super().__init__()
        self.hidden = hidden; self.layers = layers; self.dropout = dropout
        H = hidden
        self.lin_var  = nn.Linear(5, H)
        self.lin_quad = nn.Linear(5, H)
        self.lin_con  = nn.Linear(2, H)
        self.f_v_to_q = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.f_v_to_c = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.f_q_to_c = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.f_c_to_q = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.f_c_to_v = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.f_q_to_v = nn.ModuleList([nn.Linear(H, H) for _ in range(layers)])
        self.g_q1 = nn.ModuleList([MLP(H * 2, H) for _ in range(layers)])
        self.g_c  = nn.ModuleList([MLP(H * 3, H) for _ in range(layers)])
        self.g_q2 = nn.ModuleList([MLP(H * 2, H) for _ in range(layers)])
        self.g_v  = nn.ModuleList([MLP(H * 3, H) for _ in range(layers)])
        head_in = H * 3 + 3

        def _head():
            return nn.Sequential(
                nn.Linear(head_in, H), nn.ReLU(),
                nn.Dropout(dropout), nn.Linear(H, 1)
            )

        self.head_direct = _head()
        self.head_delta  = _head()

    def forward(self, data):
        h_v = self.lin_var(data["var"].x)
        h_q = self.lin_quad(data["quad"].x)
        h_c = self.lin_con(data["constr"].x)

        e_vq, w_vq = data["var", "inc", "quad"].edge_index,          data["var", "inc", "quad"].edge_attr
        e_vc, w_vc = data["var", "lin", "constr"].edge_index,        data["var", "lin", "constr"].edge_attr
        e_qc, w_qc = data["quad", "qcoef", "constr"].edge_index,     data["quad", "qcoef", "constr"].edge_attr
        e_cq, w_cq = data["constr", "rev_qcoef", "quad"].edge_index, data["constr", "rev_qcoef", "quad"].edge_attr
        e_cv, w_cv = data["constr", "rev_lin", "var"].edge_index,    data["constr", "rev_lin", "var"].edge_attr
        e_qv, w_qv = data["quad", "rev_inc", "var"].edge_index,      data["quad", "rev_inc", "var"].edge_attr

        nv = data["var"].num_nodes; nq = data["quad"].num_nodes; nc = data["constr"].num_nodes

        for t in range(self.layers):
            h_q_bar = self.g_q1[t](torch.cat([
                h_q, edge_weighted_sum(self.f_v_to_q[t](h_v), e_vq, w_vq, nq)], 1))
            h_c_new = self.g_c[t](torch.cat([
                h_c,
                edge_weighted_sum(self.f_v_to_c[t](h_v),    e_vc, w_vc, nc),
                edge_weighted_sum(self.f_q_to_c[t](h_q_bar), e_qc, w_qc, nc)], 1))
            h_q_new = self.g_q2[t](torch.cat([
                h_q_bar,
                edge_weighted_sum(self.f_c_to_q[t](h_c_new), e_cq, w_cq, nq)], 1))
            h_v_new = self.g_v[t](torch.cat([
                h_v,
                edge_weighted_sum(self.f_c_to_v[t](h_c_new), e_cv, w_cv, nv),
                edge_weighted_sum(self.f_q_to_v[t](h_q_new), e_qv, w_qv, nv)], 1))
            h_v = F.dropout(F.relu(h_v_new), p=self.dropout, training=self.training)
            h_q = F.dropout(F.relu(h_q_new), p=self.dropout, training=self.training)
            h_c = F.dropout(F.relu(h_c_new), p=self.dropout, training=self.training)

        B = int(data["var"].batch.max().item()) + 1
        pv = global_mean_pool(h_v, data["var"].batch, size=B)
        pq = global_mean_pool(h_q, data["quad"].batch, size=B) if nq > 0 else h_v.new_zeros((B, self.hidden))
        pc = global_mean_pool(h_c, data["constr"].batch, size=B) if nc > 0 else h_v.new_zeros((B, self.hidden))

        gx = data.global_x
        if gx.dim() == 1:
            gx = gx.view(-1, 3)
        if gx.size(0) != B:
            gx = gx.view(B, -1)

        g = torch.cat([pv, pq, pc, gx], dim=1)
        ld = self.head_direct(g).view(-1)
        lk = self.head_delta(g).view(-1)
        return ld, lk


# ============================================================
# 10) Train / Eval
# ============================================================

def compute_pos_weight_binary(y, device):
    pos = float((y == 1).sum()); neg = float((y == 0).sum())
    if pos <= 0:
        return None
    return torch.tensor([neg / (pos + 1e-12)], dtype=torch.float, device=device)


def train_one_epoch(model, loader, opt, device, pw_direct,
                    lam_d: float = 0.25,
                    weight_alpha: float = 0.30):
    model.train()
    total = 0.0

    for batch in loader:
        batch = batch.to(device)
        opt.zero_grad()
        ld, lk = model(batch)

        yd = batch.y_direct.view(-1).float()
        d  = batch.delta.view(-1).float()
        m  = batch.choice_mask.view(-1).float()

        w = (1.0 - weight_alpha) + weight_alpha * d.abs()

        bce_raw = F.binary_cross_entropy_with_logits(
            ld, yd, pos_weight=pw_direct, reduction="none")
        loss_direct = (bce_raw * w).mean()

        lk_safe = torch.where(torch.isfinite(lk), lk, torch.zeros_like(lk))
        w_delta = w * m
        w_delta_sum = w_delta.sum()
        loss_delta = (
            ((lk_safe - d) ** 2 * w_delta).sum() / (w_delta_sum + 1e-12)
            if w_delta_sum > 0
            else lk.new_tensor(0.0)
        )

        loss = loss_direct + lam_d * loss_delta

        if not torch.isfinite(loss):
            opt.zero_grad()
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += float(loss.item()) * yd.numel()

    return total / max(1, len(loader.dataset))


def _safe_bal_acc(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(balanced_accuracy_score(y_true, y_pred))


def _safe_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(f1_score(y_true, y_pred, average="macro", zero_division=0))


@torch.no_grad()
def evaluate_full(model, loader, device, eps_prime: float = 0.1):
    model.eval()
    yd_all, pd_all, m_all, d_all, dk_all, keys_all = [], [], [], [], [], []

    for batch in loader:
        batch = batch.to(device)
        ld, lk = model(batch)

        yd = batch.y_direct.view(-1).float()
        m  = batch.choice_mask.view(-1).float()

        yd_all.append(yd.cpu())

        ld_safe = torch.where(torch.isfinite(ld), ld, torch.zeros_like(ld))
        pd_all.append((torch.sigmoid(ld_safe) >= 0.5).float().cpu())

        m_all.append(m.cpu())
        d_all.append(batch.delta.view(-1).cpu())

        lk_safe = torch.where(torch.isfinite(lk), lk, torch.zeros_like(lk))
        dk_all.append(lk_safe.view(-1).cpu())

        if hasattr(batch, "row_key"):
            rk = batch.row_key
            if isinstance(rk, (list, tuple)):
                keys_all.extend([str(k) for k in rk])
            else:
                keys_all.extend([str(rk)] * int(yd.numel()))
        else:
            keys_all.extend(["NA"] * int(yd.numel()))

    yd  = torch.cat(yd_all).numpy()
    pd_ = torch.cat(pd_all).numpy()
    m   = torch.cat(m_all).numpy()
    d   = torch.cat(d_all).numpy()
    dk  = torch.cat(dk_all).numpy()

    acc     = float((yd == pd_).mean())
    bal_acc = _safe_bal_acc(yd.astype(int), pd_.astype(int))
    f1      = _safe_f1(yd.astype(int), pd_.astype(int))
    mse = (float(((dk - d) ** 2 * m).sum() / (m.sum() + 1e-12))
           if m.sum() > 0 else float("nan"))

    metrics = {
        "acc": acc,
        "bal_acc": bal_acc,
        "f1_macro": f1,
        "delta_mse_comparable": mse,
        "coverage_comparable": float(m.mean()),
    }

    preds = {
        "row_key":     keys_all,
        "y_true":      yd.tolist(),
        "y_pred":      pd_.tolist(),
        "choice_mask": m.tolist(),
        "delta_true":  d.tolist(),
        "delta_pred":  dk.tolist(),
    }
    return metrics, preds, yd, pd_


# ============================================================
# 11) Deterministic split  [FIX 2]
# ============================================================

def make_deterministic_splits(keys: pd.Series, train: float = 0.8, val: float = 0.1):
    """
    Hash-based deterministic split.

    Holdout usage  (call with train=0.72, val=0.08):
        train : bucket <  72   (~72%)
        val   : 72 <= bucket < 80  (~8% = 10% of training portion, no leakage)
        test  : bucket >= 80       (~20% = same as ML pipelines)

    External usage (call with train=0.9, val=0.1):
        train : bucket <  90   (~90%)
        val   : bucket >= 90   (~10%)
        test  : (empty — uses external PKL)
    """
    h = pd.util.hash_pandas_object(keys.astype(str), index=False).to_numpy(dtype=np.uint64)
    b = (h % 100).astype(np.int64)
    tc = int(train * 100)
    vc = int((train + val) * 100)
    return (np.where(b < tc)[0],
            np.where((b >= tc) & (b < vc))[0],
            np.where(b >= vc)[0])


def pick_split_key(df: pd.DataFrame) -> pd.Series:
    for col in ["id_global", "id", "instance_id", "group_id", "size_tag", "__match_key__"]:
        if col in df.columns and df[col].notna().any():
            return df[col].astype(str)
    return pd.Series(np.arange(len(df)), index=df.index).astype(str)


# ============================================================
# 12) Saving helpers
# ============================================================

def save_confusion_matrix(y_true, y_pred, labels, run_dir: Path, fname: str, title: str):
    cm = confusion_matrix(y_true.astype(int), y_pred.astype(int), labels=list(range(len(labels))))
    disp = ConfusionMatrixDisplay(cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(7, 5))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(run_dir / fname, dpi=200)
    plt.close()

    cm_df = pd.DataFrame(cm,
        index=[f"true_{l}" for l in labels],
        columns=[f"pred_{l}" for l in labels])
    cm_df.to_csv(run_dir / fname.replace(".png", ".csv"), index=True)
    print(f"[{_ts()}] [save] {fname}")


def save_epoch_log(epoch_log: List[dict], run_dir: Path, tag: str):
    pd.DataFrame(epoch_log).to_csv(run_dir / f"epoch_log_{tag}.csv", index=False)
    print(f"[{_ts()}] [save] epoch_log_{tag}.csv")


def save_results(metrics: dict, run_dir: Path, tag: str, split: str, cfg: RunConfig):
    row = {
        "split": split,
        "run_tag": tag,
        **metrics,
        "train_pkl": cfg.train_pkl,
        "test_pkl": cfg.test_pkl or "holdout",
        "epochs": cfg.epochs,
        "hidden": cfg.hidden,
        "layers": cfg.layers,
        "dropout": cfg.dropout,
        "lr": cfg.lr,
        "eps_prime": cfg.eps_prime,
        "tau_sign": cfg.tau_sign,
        "epsilon_gap": cfg.epsilon_gap,
        "weight_alpha": cfg.weight_alpha,
        "gap_formula": "fixed_additive",
        "decision_rule": "lp_biased",
        "comparable_def": "finite_not_unbounded_not_excluded",
    }
    pd.DataFrame([row]).to_csv(run_dir / f"results_gnn_{tag}.csv", index=False)
    print(f"[{_ts()}] [save] results_gnn_{tag}.csv")


def save_predictions(preds: dict, run_dir: Path, tag: str, split: str):
    df = pd.DataFrame(preds)
    df.insert(0, "split", split)
    df.to_csv(run_dir / f"preds_{tag}.csv", index=False)
    print(f"[{_ts()}] [save] preds_{tag}.csv")


def save_config(cfg: RunConfig, run_dir: Path, tag: str):
    pd.DataFrame([asdict(cfg)]).to_csv(run_dir / f"config_{tag}.csv", index=False)
    print(f"[{_ts()}] [save] config_{tag}.csv")


# ============================================================
# 13) main()  [FIX 2 — split corrected]
# ============================================================

def main(run_cfg: Optional[RunConfig] = None):
    if run_cfg is None:
        run_cfg = RunConfig()

    if abs(float(run_cfg.eps_prime) - float(run_cfg.tau_sign)) > 1e-12:
        raise ValueError(
            f"eps_prime ({run_cfg.eps_prime}) and tau_sign ({run_cfg.tau_sign}) "
            f"must match for alignment with the regression code.")

    np.random.seed(0); torch.manual_seed(0)

    run_dir, tag = _make_run_dir(run_cfg)
    print(f"[{_ts()}] ══════════════════════════════════════════")
    print(f"[{_ts()}] GNN CLASSIFIER (fixed)  tag={tag}")
    print(f"[{_ts()}] Output dir: {run_dir.resolve()}")
    print(f"[{_ts()}] gap formula   : fixed additive")
    print(f"[{_ts()}] decision rule : LP-biased (delta > -eps_prime)")
    print(f"[{_ts()}] comparable    : finite + not unbounded + not excluded [FIX 1]")
    print(f"[{_ts()}] epsilon_gap={run_cfg.epsilon_gap}")
    print(f"[{_ts()}] tau_sign={run_cfg.tau_sign}  eps_prime={run_cfg.eps_prime}")
    print(f"[{_ts()}] weight_alpha={run_cfg.weight_alpha}")
    print(f"[{_ts()}] Split (holdout): 72/8/20 — test=same 20% as ML [FIX 2]")
    print(f"[{_ts()}] ══════════════════════════════════════════")

    gcfg = GraphConfig(
        tol=run_cfg.tol,
        eps_prime=run_cfg.eps_prime,
        tau_sign=run_cfg.tau_sign,
        epsilon_gap=run_cfg.epsilon_gap,
        max_quad_nodes=None if run_cfg.max_quad_nodes <= 0 else int(run_cfg.max_quad_nodes),
        max_lin_edges_per_constr=None if run_cfg.max_lin_edges_per_constr <= 0 else int(run_cfg.max_lin_edges_per_constr),
    )
    mit = None if run_cfg.max_items_train <= 0 else int(run_cfg.max_items_train)
    mie = None if run_cfg.max_items_test  <= 0 else int(run_cfg.max_items_test)

    train_ds_full = QCQPTripartiteDataset(
        run_cfg.train_pkl, run_cfg.train_csv, gcfg,
        cache_dir=run_cfg.cache_dir, max_items=mit, require_labels=True,
        filter_size_tag=run_cfg.train_size_tag, filter_n=run_cfg.train_n,
        filter_m_total=run_cfg.train_m_total,
        verbose=run_cfg.verbose, allow_row_order_fallback=run_cfg.allow_row_order_fallback)

    if len(train_ds_full) == 0:
        raise RuntimeError("Train dataset empty. Check PKL/CSV and keys.")

    # [FIX 2] Split logic:
    #   Holdout : 72/8/20  -> test = bucket>=80 (same as ML pipelines), val inside train portion
    #   External: 90/10    -> test = external PKL; val from training PKL
    if run_cfg.test_pkl:
        # External test: 90% train / 10% val from training PKL
        tr_idx, va_idx, _ = make_deterministic_splits(
            pick_split_key(train_ds_full.df), 0.9, 0.1)
        test_ds = QCQPTripartiteDataset(
            run_cfg.test_pkl, run_cfg.test_csv, gcfg,
            cache_dir=None, max_items=mie, require_labels=True,
            filter_size_tag=run_cfg.test_size_tag, filter_n=run_cfg.test_n,
            filter_m_total=run_cfg.test_m_total,
            verbose=run_cfg.verbose, allow_row_order_fallback=run_cfg.allow_row_order_fallback)
        if len(test_ds) == 0:
            raise RuntimeError("Test dataset empty.")
        test_split_name = "external_test"
        print(f"[{_ts()}] [split] External: train=90% val=10% of training PKL", flush=True)
    else:
        # Holdout: train=72%, val=8% (10% of train), test=20% (same as ML)
        tr_idx, va_idx, te_idx = make_deterministic_splits(
            pick_split_key(train_ds_full.df), 0.72, 0.08)
        test_ds = Subset(train_ds_full, te_idx.tolist())
        test_split_name = "holdout_test"
        print(f"[{_ts()}] [split] Holdout: train={len(tr_idx)} val={len(va_idx)} "
              f"test={len(te_idx)} (bucket>=80, same as ML)", flush=True)

    train_ds = Subset(train_ds_full, tr_idx.tolist())
    val_ds   = Subset(train_ds_full, va_idx.tolist())
    print(f"[{_ts()}] [data] train={len(train_ds)} | val={len(val_ds)} | test={len(test_ds)}", flush=True)

    train_loader = DataLoader(train_ds, batch_size=run_cfg.batch_train, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=run_cfg.batch_eval,  shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=run_cfg.batch_eval,  shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[{_ts()}] [device] {device}", flush=True)

    df_tr = train_ds_full.df.iloc[tr_idx]
    pw_direct = compute_pos_weight_binary(df_tr["__y_direct__"].to_numpy(float), device)
    print(f"[{_ts()}] [weights] pw_direct={None if pw_direct is None else round(float(pw_direct), 4)}"
          f"  weight_alpha={run_cfg.weight_alpha}")

    model = TripartiteTermGNN(run_cfg.hidden, run_cfg.layers, run_cfg.dropout).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=run_cfg.lr, weight_decay=run_cfg.weight_decay)

    epoch_log = []
    for epoch in range(1, run_cfg.epochs + 1):
        loss = train_one_epoch(model, train_loader, opt, device, pw_direct,
                               run_cfg.lambda_delta, run_cfg.weight_alpha)
        vm, _, _, _ = evaluate_full(model, val_loader, device, run_cfg.eps_prime)
        row = {"epoch": epoch, "train_loss": loss, **{f"val_{k}": v for k, v in vm.items()}}
        epoch_log.append(row)
        print(f"[{_ts()}] Epoch {epoch:02d} | loss={loss:.4f} | "
              f"acc={vm['acc']:.4f} | bal_acc={vm['bal_acc']:.4f} | "
              f"f1_macro={vm['f1_macro']:.4f} | "
              f"delta_mse_cmp={vm['delta_mse_comparable']:.6f} | "
              f"cov={vm['coverage_comparable']:.3f}", flush=True)

    save_epoch_log(epoch_log, run_dir, tag)

    test_metrics, preds, yd, pd_ = evaluate_full(model, test_loader, device, run_cfg.eps_prime)
    print(f"\n[{_ts()}] TEST: {test_metrics}", flush=True)

    save_predictions(preds, run_dir, tag, test_split_name)
    save_confusion_matrix(yd, pd_, ["SDP(0)", "LP(1)"],
                          run_dir, f"cm_{tag}.png", "Direct LP/SDP Prediction")
    save_results(test_metrics, run_dir, tag, test_split_name, run_cfg)
    save_config(run_cfg, run_dir, tag)

    g0 = train_ds_full[0]
    print(f"\n[{_ts()}] [graph sanity]")
    print(f"  var    : {g0['var'].num_nodes} nodes  shape={list(g0['var'].x.shape)}")
    print(f"  quad   : {g0['quad'].num_nodes} nodes  shape={list(g0['quad'].x.shape)}")
    print(f"  constr : {g0['constr'].num_nodes} nodes  shape={list(g0['constr'].x.shape)}")
    for et in g0.edge_types:
        print(f"  {et}: E={g0[et].edge_index.shape[1]}  attr={list(g0[et].edge_attr.shape)}")

    written = sorted(p.name for p in run_dir.glob("*"))
    print(f"\n[{_ts()}] Files written ({len(written)}): {written}")

    return model, test_metrics


if __name__ == "__main__":
    cfg = RunConfig(
        epsilon_gap=1e-3,
        tau_sign=0.1,
        eps_prime=0.1,
        weight_alpha=0.30,
    )
    model, metrics = main(cfg)